# Explore Lab data

A first look at the lab data table (RPSD and Kaiser) before analysis.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type)
2. **Drop** columns that are completely empty
3. **Review** columns that hold only a single value
4. **Parse** the date columns (currently stored as text)
5. **Tidy** column types
6. **Save** a cleaned copy + keep an audit note of what changed

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.

> **Before running:** finish the venv setup in this repo and install the packages — open a terminal and run `pip install pandas pyarrow`. The `requirements.txt` alongside this notebook lists everything.
>
> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [19]:
from pathlib import Path
import pandas as pd

In [20]:
#setting display options to show all columns and full width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Allow full width

In [21]:
# --- RPSD import ---
rpsd_PATH = Path("../data/2024_Embryology_Assurance_RPSD.csv")

# loading data
rpsd_df = pd.read_csv(rpsd_PATH)

# --- Kaiser import ---
kaiser_PATH = Path("../data/2024_Embryology_Assurance_Kaiser.csv")

# loading data
kaiser_df = pd.read_csv(kaiser_PATH)

# --- 2 techs import ---
techs2_PATH = Path("../data/2024_Embryology_Assurance_2_techs.csv")

# loading data
techs2_df = pd.read_csv(techs2_PATH)

## Name Cleaning

In [22]:
# rename certain Kaiser columns to match RPSD columns
kaiser_df = kaiser_df.rename(columns={
    "# of Follicle": "# of Follicles",
    "Clomiphne dosage": "Clomiphene dosage",
    "Lupron and Pregnyl trigger": "Lupon & Pregnyl Trigger",
    "Menopour dosage": "Menopur dosage"
})

In [23]:
# looking to see if all the column names are the same in all df's

dfs = {
    "RPSD": rpsd_df,
    "Kaiser": kaiser_df,
    "2 Techs": techs2_df
}

# Reference columns
reference_name, reference_df = next(iter(dfs.items()))
reference_cols = set(reference_df.columns)

for name, df in dfs.items():
    cols = set(df.columns)

    missing = reference_cols - cols
    extra = cols - reference_cols

    print(f"\n{name}")
    if not missing and not extra:
        print("✓ Columns match the reference.")
    else:
        if missing:
            print("Missing:", sorted(missing))
        if extra:
            print("Extra:", sorted(extra))


RPSD
✓ Columns match the reference.

Kaiser
Missing: ['FDA']

2 Techs
Missing: [' D5 Abnormal F', ' D5 Abnormal G', ' D5 Abnormal P', ' D5 Normal F', ' D5 Normal G', ' D5 Normal P', ' D6 Abnormal F', ' D6 Abnormal G', ' D6 Abnormal P', ' D6 Normal F', ' D6 Normal G', ' D6 Normal P', ' D7 Abnormal F', ' D7 Abnormal G', ' D7 Abnormal P', ' D7 Normal F', ' D7 Normal G', ' D7 Normal P', '# 0PN to Eploid blast', '# 1PN to Euploid blast', '# Complex ABN', '# Egg retrieved/Thawed', '# Mosaic', '# no signal ', '# of EMB ET', '# of Follicles', '# of Sacs', '#0PN to blast', '#1PN to blast', '#EMB Survived', '#EMB Thawed', 'AH after Thaw', 'AMH Level', 'Age', 'Biochem', 'Bx Tech for FET', 'CC', 'Chaotic', 'Clomiphene dosage', 'Comments', 'D5 Bx Tech', 'D5 FRZ Tech', 'D6 Bx Tech', 'D6 FRZ Tech', 'D7 Bx Tech', 'D7 FRZ Tech', 'Donor Eggs', 'Dose Increased', 'Double Lumen', 'EGG FRZ Tech', 'ER Dr.', 'ER Tech', 'ET Date', 'ET Dr.', 'ET Tech', 'ET Type', 'Egg FRZ', 'Endo Thickness', 'Endometrial prep 

## Filtering to IVF cases and FET cases in separate dfs

### first identifying IVF cases

In [24]:
# first adding a clinic column to fill in RPSD or Kaiser
rpsd_df['clinic'] = 'RPSD'
kaiser_df['clinic'] = 'Kaiser'

In [25]:
# printing all values for "procedure"
display(rpsd_df['Procedure'].unique())

<StringArray>
[                      'FET',                 'IVF Cycle',
               'Thaw for Bx',             'Oocyte Freeze',
                  'FET w/GC', 'IVF Cycle-->Oocyte Freeze',
               'Oocyte Thaw',                         nan,
                 'Thaw Rebx',                   'Thaw Bx',
             'Thaw for Rebx',                  'Thaw RBX',
                 'FET w/ GC',              'Oocyte thaw ',
                 'IVF cycle',               'Oocyte thaw',
             'Oocyte freeze',             'Co-incubation',
                'IVF cycle ',            'Oocyte Freeze ',
             'Ooctye Freeze',                 'FET W/ GC']
Length: 22, dtype: str

In [26]:
# RPSD procedure value can be "IVF cycle ", "IVF cycle", or "IVF Cycle"
rpsd_ivf_df = rpsd_df[rpsd_df["Procedure"].str.strip().str.lower() == "ivf cycle"]

#Kaiser procedure value can be "IVF Cycle" or "IVF cycle"
kaiser_ivf_df = kaiser_df[kaiser_df["Procedure"].str.strip().str.lower() == "ivf cycle"]

In [27]:
display(rpsd_ivf_df.shape)
display(kaiser_ivf_df.shape)

(395, 102)

(128, 101)

In [28]:
#combining rpsd and kaiser ivf df's
ivf_df = pd.concat([rpsd_ivf_df, kaiser_ivf_df], ignore_index=True)

display(ivf_df.shape)
display(ivf_df.head())

(523, 102)

,Cycle #,Patient Name,Procedure,ER Date,Age,Donor Eggs,FDA,AMH Level,Primary Dr.,STIM D3 E2,Peak E2,Dose Increased,Menopur dosage,Follistim dosage,Clomiphene dosage,Ganirellx,Indomethacin,Lupon & Pregnyl Trigger,Endo Thickness,P4 Level,# of Follicles,# Egg retrieved/Thawed,Double Lumen,ER Dr.,ER Tech,Hyase Tech,TESA/TESE,Zymot,SW Tech,Egg FRZ,ICSI hours post ER,ICSI Tech,# Mature eggs,CC,# 2PN,# Lysed,# 0PN,# 1PN,# 3PN,# of Bx,D5 Normal G,D5 Normal F,D5 Normal P,D5 Abnormal G,D5 Abnormal F,D5 Abnormal P,D6 Normal G,D6 Normal F,D6 Normal P,D6 Abnormal G,D6 Abnormal F,D6 Abnormal P,D7 Normal G,D7 Normal F,D7 Normal P,D7 Abnormal G,D7 Abnormal F,D7 Abnormal P,# Complex ABN,# Mosaic,Chaotic,# no signal,#0PN to blast,# 0PN to Eploid blast,#1PN to blast,# 1PN to Euploid blast,# of D5 usable blast,# of D6 usable blast,# of D7 usable blast,PGT,EGG FRZ Tech,D5 Bx Tech,D5 FRZ Tech,D6 Bx Tech,D6 FRZ Tech,D7 Bx Tech,D7 FRZ Tech,Reasons for No ET,FET Freeze Date,ET Date,FET w/PGT,Bx Tech for FET,Endometrial prep protocol,Thaw Tech,Recover hours after thaw,FRexpnded at time of ET,#EMB Thawed,#EMB Survived,AH after Thaw,ET Type,Quality of EMB transferred,Primary Dr..1,ET Dr.,ET Tech,# of EMB ET,HCG>5,# of Sacs,Ongoing,Biochem,SAB,Comments,clinic
0,#24-003,"Sridhar, Priya",IVF Cycle,1/4/2024,39.0,N,NaN,0.441,STAN,48,2016.0,Y,3825.0,3925.0,1100,NaN,NaN,NaN,3.4,0.33,7.0,7.0,Y,FLO,KON,TIN,N,N,KUC,N,3.5,ZHA,4.0,N,1.0,NaN,3.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,Y,NaN,NaN,NaN,ZHA,KIM,NaN,NaN,No normal blast,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RPSD
1,#24-008,"Morton, Nissa",IVF Cycle,1/8/2024,41.0,N,NaN,1.61,AGA,142,4070.0,N,1350.0,2025.0,1000,NaN,NaN,NaN,8.7,6.48,21.0,28.0,N,SU,KON,KON,N,Y,KON,N,5,ZHA,13.0,N,6.0,NaN,6.0,NaN,1.0,6.0,1.0,NaN,NaN,1.0,2.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,2.0,NaN,Y,NaN,ZHA,ARA,ZHA,ARA,NaN,NaN,Embryo banking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RPSD
2,#24-012,"Curtis, Megan",IVF Cycle,1/8/2024,35.0,N,NaN,2.02,SU,244,5775.0,N,1350.0,1350.0,NaN,Y,NaN,NaN,7.2,10.28,9.0,22.0,N,DUL,KON,KIM,N,Y,ARA,N,4.5,KIM,19.0,Y,14.0,NaN,2.0,3.0,NaN,10.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,5,5.0,NaN,Y,NaN,ZHA,ARA,KIM,ARA,NaN,NaN,Embryo banking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RPSD
3,#24-013,"Limon, Noerena",IVF Cycle,1/9/2024,41.0,N,NaN,4.24,AGA,69,5942.0,N,825.0,1025.0,800,NaN,NaN,NaN,10.2,10.78,11.0,37.0,N,DUL,KON,ARA,N,Y,CRU,N,5,ZHA,27.0,N,18.0,7.0,2.0,NaN,NaN,4.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3.0,NaN,Y,NaN,ZHA,ARA,KIM,ARA,NaN,NaN,No normal blast,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RPSD
4,#24-014,"Cesare, Catherine",IVF Cycle,1/9/2024,32.0,N,NaN,1.88,GAR,71,1445.0,Y,2400.0,2400.0,1100,Y,NaN,NaN,7.4,NaN,6.0,20.0,N,GAR,KON,KIM,N,Y,CRU,N,4,ZHA,14.0,N,11.0,1.0,2.0,NaN,NaN,10.0,3.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,2.0,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3,6.0,1.0,Y,NaN,ZHA,ARA,KIM,ARA,KIM,TIN,Embryo banking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RPSD


In [29]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how full it is, how many distinct values, its dtype,
    and whether all non-null values are identical."""
    out = pd.DataFrame({
        "non_null": frame.notna().sum(),
        "nulls": frame.isna().sum(),
        "distinct": frame.nunique(dropna=True),
        "dtype": frame.dtypes.astype(str),
    })

    out["pct_null"] = (out["nulls"] / len(frame) * 100).round(1)

    # True if all non-null values in the column are the same
    out["all_values_same"] = out["distinct"] <= 1

    return out.sort_values("non_null", ascending=False)

ivf_prof = profile(ivf_df)
ivf_prof

,non_null,nulls,distinct,dtype,pct_null,all_values_same
Cycle #,523,0,523,str,0.0,False
Patient Name,523,0,452,str,0.0,False
Procedure,523,0,3,str,0.0,False
ER Date,523,0,245,str,0.0,False
Age,523,0,23,float64,0.0,False
...,...,...,...,...,...,...
HCG>5,0,523,0,str,100.0,True
Ongoing,0,523,0,str,100.0,True
# of Sacs,0,523,0,float64,100.0,True
SAB,0,523,0,str,100.0,True


In [30]:
# Keep any embryo outcome columns even if they are constant
keep_cols = [
    c for c in ivf_df.columns
    if ("D5" in c or "D6" in c or "D7" in c)
    and ("Normal" in c or "Abnormal" in c)
]

# Drop only empty/constant columns that are NOT in keep_cols
ivf_cols_to_drop = [
    c for c in ivf_prof.index[
        (ivf_prof["non_null"] == 0) | (ivf_prof["all_values_same"])
    ]
    if c not in keep_cols
]

print(f"Dropping {len(ivf_cols_to_drop)} empty or constant columns:\n")
for c in ivf_cols_to_drop:
    print("  -", c)

ivf_df = ivf_df.drop(columns=ivf_cols_to_drop)

print(f"\nRemaining: {ivf_df.shape[1]} columns")

Dropping 23 empty or constant columns:

  - #1PN to blast
  - FET Freeze Date
  - ET Date
  - Endometrial prep protocol
  - Thaw Tech
  - FET w/PGT
  - Bx Tech for FET
  - FRexpnded at time of ET
  - Recover hours after thaw
  - #EMB Thawed
  - #EMB Survived
  - Quality of EMB transferred
  - Primary Dr..1
  - AH after Thaw
  - ET Type
  - ET Tech
  - ET Dr.
  - # of EMB ET
  - HCG>5
  - Ongoing
  - # of Sacs
  - SAB
  - Biochem

Remaining: 79 columns


In [31]:
# saving as csv
ivf_df.to_csv("../data/ivf_lab_data_cleaned.csv", index=False)

### now identifying FET cases

In [32]:
# RPSD procedure value is only "FET" unless I also want to include gestational carriers
rpsd_fet_df = rpsd_df[rpsd_df["Procedure"].str.strip().str.lower() == "fet"]

#Kaiser procedure value can be FET
kaiser_fet_df = kaiser_df[kaiser_df["Procedure"].str.strip().str.lower() == "fet"]

In [33]:
display(rpsd_fet_df.shape)
display(kaiser_fet_df.shape)

(409, 102)

(147, 101)

In [34]:
#combining rpsd and kaiser fet df's
fet_df = pd.concat([rpsd_fet_df, kaiser_fet_df], ignore_index=True)

display(fet_df.shape)
display(fet_df.head())

(556, 102)

,Cycle #,Patient Name,Procedure,ER Date,Age,Donor Eggs,FDA,AMH Level,Primary Dr.,STIM D3 E2,Peak E2,Dose Increased,Menopur dosage,Follistim dosage,Clomiphene dosage,Ganirellx,Indomethacin,Lupon & Pregnyl Trigger,Endo Thickness,P4 Level,# of Follicles,# Egg retrieved/Thawed,Double Lumen,ER Dr.,ER Tech,Hyase Tech,TESA/TESE,Zymot,SW Tech,Egg FRZ,ICSI hours post ER,ICSI Tech,# Mature eggs,CC,# 2PN,# Lysed,# 0PN,# 1PN,# 3PN,# of Bx,D5 Normal G,D5 Normal F,D5 Normal P,D5 Abnormal G,D5 Abnormal F,D5 Abnormal P,D6 Normal G,D6 Normal F,D6 Normal P,D6 Abnormal G,D6 Abnormal F,D6 Abnormal P,D7 Normal G,D7 Normal F,D7 Normal P,D7 Abnormal G,D7 Abnormal F,D7 Abnormal P,# Complex ABN,# Mosaic,Chaotic,# no signal,#0PN to blast,# 0PN to Eploid blast,#1PN to blast,# 1PN to Euploid blast,# of D5 usable blast,# of D6 usable blast,# of D7 usable blast,PGT,EGG FRZ Tech,D5 Bx Tech,D5 FRZ Tech,D6 Bx Tech,D6 FRZ Tech,D7 Bx Tech,D7 FRZ Tech,Reasons for No ET,FET Freeze Date,ET Date,FET w/PGT,Bx Tech for FET,Endometrial prep protocol,Thaw Tech,Recover hours after thaw,FRexpnded at time of ET,#EMB Thawed,#EMB Survived,AH after Thaw,ET Type,Quality of EMB transferred,Primary Dr..1,ET Dr.,ET Tech,# of EMB ET,HCG>5,# of Sacs,Ongoing,Biochem,SAB,Comments,clinic
0,#24-001,"Dudley, Sarah",FET,NaN,37.0,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9/5/2023,1/3/2024,Y,KIM,Transdermal Patches,TIN,4.0,Y,1.0,1.0,Y,FET-D5,XBAAG,FLOR,FLOR,ZHA,1.0,Y,1.0,Y,N,N,NaN,RPSD
1,#24-002,"Sahu, Somya",FET,NaN,36.0,N,ED/NE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10/3/2023,1/3/2024,Y,ZHA,Transdermal Patches,KIM,4.0,Y,1.0,1.0,Y,FET-D5,FBBBF,AGA,AGA,ZHA,1.0,Y,1.0,Y,N,N,NaN,RPSD
2,#24-004,"Kadkhodaeigolmakani, Marzieh",FET,NaN,35.0,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3/20/2020,1/4/2024,Y,ZHA,Transdermal Patches,TIN,4.0,Y,1.0,1.0,N,FET-D6,HBBBF,DUL,DEL,KIM,1.0,Y,1.0,Y,N,N,NaN,RPSD
3,#24-005,"Needham, Perri",FET,NaN,35.0,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11/30/2022,1/4/2024,Y,ZHA,Transdermal Patches,TIN,3.5,Y,1.0,1.0,N,FET-D6,HBBCP,STAN,FLOR,TOR,1.0,N,0.0,N,N,N,NaN,RPSD
4,#24-006,"Patil, Kavya",FET,NaN,32.0,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10/31/2017,1/5/2024,Y,ZHA,Natural,TOR,3.0,Y,1.0,1.0,Y,FET-D6,HBABG,DUL,DUL,TOR,1.0,Y,1.0,Y,N,N,NaN,RPSD


In [35]:
# profiling the fet df
fet_prof = profile(fet_df)
fet_prof

# dropping columns with all the same value
fet_cols_to_drop = fet_prof.index[
    (fet_prof["non_null"] == 0) | (fet_prof["all_values_same"])
].tolist()

print(f"Dropping {len(fet_cols_to_drop)} empty or constant columns:\n")
for c in fet_cols_to_drop:
    print("  -", c)

fet_df = fet_df.drop(columns=fet_cols_to_drop)

print(f"\nRemaining: {fet_df.shape[1]} columns")

Dropping 73 empty or constant columns:

  - Procedure
  - Double Lumen
  - AMH Level
  - Primary Dr.
  - STIM D3 E2
  - Peak E2
  - Dose Increased
  - Menopur dosage
  - Follistim dosage
  - Reasons for No ET
  - D5 FRZ Tech
  - D6 Bx Tech
  - # 0PN
  - CC
  - # Mature eggs
  - ICSI Tech
  - # 2PN
  - Egg FRZ
  - SW Tech
  - Zymot
  - TESA/TESE
  - Hyase Tech
  - ER Tech
  - ER Dr.
  - ICSI hours post ER
  - # Egg retrieved/Thawed
  - # of Follicles
  - P4 Level
  - Endo Thickness
  - Lupon & Pregnyl Trigger
  - Indomethacin
  - Ganirellx
  - Clomiphene dosage
  - # 3PN
  - # of Bx
  -  D5 Abnormal F
  -  D5 Normal P
  -  D6 Abnormal F
  -  D6 Abnormal G
  - # of D5 usable blast
  - # Complex ABN
  - D6 FRZ Tech
  - # of D6 usable blast
  - PGT
  - D5 Bx Tech
  - # 1PN
  - # Lysed
  - ER Date
  - # Mosaic
  - # of D7 usable blast
  - # 0PN to Eploid blast
  - #1PN to blast
  - # 1PN to Euploid blast
  - #0PN to blast
  -  D6 Abnormal P
  -  D7 Normal G
  -  D7 Normal F
  -  D5 Normal G

In [36]:
# saving as csv
fet_df.to_csv("../data/fet_lab_data_cleaned.csv", index=False)